In [ ]:
from __future__ import annotations
import numpy as np
import random
import copy
import importlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import imageio_ffmpeg
import pandas as pd

from pathlib import Path
from typing import Tuple, List
from dataclasses import replace
from numpy import array, zeros

# from Big_Class import Big_Class  # already imported one NETfuncs is imported
from User_Variables import User_Variables  # already imported one NETfuncs is imported
from Network_Structure import Network_Structure  # already imported one NETfuncs is imported
from Big_Class import Big_Class
from Network_State import Network_State
from Networkx_Net import Networkx_Net
import matrix_functions, functions, plot_funcs, solve, colors, file_funcs
import statistical_analysis as statistics

# for vids
mpl.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()

## colors

In [ ]:
importlib.reload(colors)

colors_lst, red, cmap = colors.color_scheme()
cmap

plt.rcParams['axes.prop_cycle'] = plt.cycler('color', colors_lst)

# Set up Network

In [ ]:
## Strcture, Variables and Supervisor
importlib.reload(matrix_functions)
importlib.reload(plot_funcs)

import config
importlib.reload(config)
CFG = config.CFG

import Network_Structure as network_structure_module
import User_Variables as user_variables_module
import Supervisor_Class as supervisor_module
import Network_State as network_state_module
importlib.reload(user_variables_module)
importlib.reload(supervisor_module)
importlib.reload(network_structure_module)
importlib.reload(network_state_module)
from Network_Structure import Network_Structure
from User_Variables import User_Variables
from Supervisor_Class import Supervisor
from Network_State import Network_State

## Structure is the first simulation class instance
Strctr = Network_Structure(CFG)
if CFG.Sprvsr.control == 'current' and CFG.Sprvsr.training_scheme in {"grad_desc", "BEASTAL_NTC"}:  # only current NTCs need inputs to ground
    Strctr.build_incidence(Strctr.net_type, in_to_g = True)
else:
    Strctr.build_incidence(Strctr.net_type)  
BigClass = Big_Class(Strctr)

print('input_nodes_arr ', Strctr.input_nodes_arr)
print('extraInput_nodes_arr ', Strctr.extraInput_nodes_arr)
print('inter_nodes_arr ', Strctr.inter_nodes_arr)
print('output_nodes_arr ', Strctr.output_nodes_arr)
print('extraOutput_nodes_arr ', Strctr.extraOutput_nodes_arr)
print('ground_nodes_arr ', Strctr.ground_nodes_arr)

Variabs = User_Variables(CFG, Strctr)
BigClass.add_Variabs(Variabs)

Sprvsr = Supervisor(CFG, Strctr, Variabs)
BigClass.add_Sprvsr(Sprvsr)

State = Network_State(CFG, BigClass)
BigClass.add_State(State)  # add to big class

In [ ]:
# Incidence matrices, actual and fictitious (for networks with hidden)

print('U', Strctr.DM)

if Sprvsr.training_scheme in ['BEASTAL', 'BEASTAL_NTC', 'PIECETAL_NTC', 'Adjoint_pressure']:  # only BEASTAL uses p-inv U
    Strctr.build_inverse_incidence()
    print('U_dagger', Strctr.DM_dagger)


# build fictitious Network_Structure as if no inter nodes
if Sprvsr.training_scheme in {'BEASTAL', 'BEASTAL_NTC'} and Strctr.Ninter > 0:
    fict_cfg = replace(CFG, Strctr=replace(CFG.Strctr, Ninter=0))
    fict_cfg = replace(fict_cfg, Strctr=replace(fict_cfg.Strctr, net_type="FC"))
    Strctr_fict = Network_Structure(fict_cfg)
    Strctr_fict.build_incidence(Strctr_fict.net_type)
    Strctr_fict.build_inverse_incidence()
    BigClass.add_Strctr_fict(Strctr_fict)  # add to big class

# print('grad_loss_vec', Sprvsr.grad_loss_vec)

In [ ]:
CM_dagger = Strctr.DM.T@Strctr.DM@Strctr.DM_dagger
CM_dagger

In [ ]:
## Networkx Net

import Networkx_Net as networkx_net_module
importlib.reload(networkx_net_module)
from Networkx_Net import Networkx_Net

## build network graphics class and plot structure
NET = Networkx_Net(CFG)
NET.buildNetwork(BigClass)
NET.build_pos_lattice(BigClass, plot=True, node_labels=False)
BigClass.add_NET(NET)  # add to big class

## Constant vals

In [ ]:
# if Sprvsr.control != "current":
#     raise ValueError("Set CFG.Sprvsr.control to 'current' and rerun the setup cells")
# State.initiate_resistances(BigClass, CFG.State.R_vec_i, add_noise=CFG.State.R_noise)
# State.t = 0

# State.initiate_accuracy_vec(BigClass)
# State.input_drawn = np.zeros(Strctr.Nin)
# State.input_drawn[0] = 0.11 # injected current at the first input node; ground supplies the return current
# State.extraInput = np.zeros(Strctr.extraNin)
# total_t = 50
# iters = int(total_t/Variabs.dt_upper)
# t_vec = np.arange(iters + 1) * Variabs.dt_upper
# i_in_t = np.zeros([iters, Strctr.NE])
# for i in range(iters):
#     State.solve_flow_given_modality(BigClass, "measure")
#     State.update_Rs(BigClass)
#     State.t += 1
#     i_in_t[i] = State.u
# State.solve_flow_given_modality(BigClass, "measure")  # final voltages after the last resistance update

# fig, (ax_R, ax_i, ax_T) = plt.subplots(3, 1, sharex=True, figsize=(7, 8))
# ax_R.set_prop_cycle(plt.cycler(color=colors_lst))
# ax_R.plot(t_vec, State.R_in_t)
# ax_R.set_ylabel(r'$R$')
# ax_R.set_yscale('log')
# ax_R.set_ylim([2e-1, 1.5e3])
# ax_i.set_prop_cycle(plt.cycler(color=colors_lst))
# ax_i.plot(t_vec[1:], i_in_t)
# ax_i.set_ylabel(r'$i$')
# ax_i.set_ylim([-0.02, 0.22])
# ax_T.set_prop_cycle(plt.cycler(color=colors_lst))
# ax_T.plot(t_vec, State.T_in_t)
# ax_T.set_xlabel(r'$t$')
# ax_T.set_ylabel(r'$T$')
# ax_T.set_ylim([290, 420])
# ax_i.axhline(0.15, color=red, linestyle='--')
# ax_T.axhline(398, color=red, linestyle='--')
# plt.tight_layout()

In [ ]:
# import sys
# sys.exit("program complete, Exiting kernel.")

In [ ]:
print('Q_max = ', Sprvsr.Q_max)
print('desired output = ', Sprvsr.desired_in_t[-3:])
# Sprvsr.T_max

# Train

## Define loop

In [ ]:
update_iters = 1

def compare_BEASTAL_to_GD(reference_State: Network_State) -> tuple[np.ndarray, np.ndarray]:
    """Return BEASTAL and finite-difference GD resistance steps for all alternating-one inputs."""
    batch_cfg = replace(CFG, Sprvsr=replace(CFG.Sprvsr, batch_size=Strctr.Nin, iterations=1, dataset_type="alternating ones", anneal_alpha=False))
    batch_Sprvsr = Supervisor(batch_cfg, Strctr, Variabs)
    batch_Sprvsr.reset_training_history(Strctr)
    batch_Sprvsr.alpha = Sprvsr.alpha
    batch_Sprvsr.update_vec = Sprvsr.update_vec.copy()
    batch_State = copy.deepcopy(reference_State)
    batch_State.t = 0
    trained_State, trained_Sprvsr = BigClass.State, BigClass.Sprvsr
    BigClass.add_State(batch_State)
    BigClass.add_Sprvsr(batch_Sprvsr)
    try:
        R_before_update = batch_State.R_in_t[-1].copy()
        delta_K_GD = np.zeros(Strctr.NE)
        delta_R_GD = np.zeros(Strctr.NE)
        for b in range(Strctr.Nin):
            batch_State.draw_input_and_desired(batch_Sprvsr, b)
            batch_State.solve_flow_given_modality(BigClass, "measure")
            if Variabs.grad_desc_use == 'K':
                delta_K_sample, _ = batch_State.dK_grad_desc(Strctr, Variabs.dK_step, batch_State.desired, func='MSE')
                delta_K_GD += delta_K_sample
            elif Variabs.grad_desc_use == 'R':
                delta_R_sample, _ = batch_State.dR_grad_desc(Strctr, Variabs.dR_step, batch_State.desired, func='MSE')
                delta_R_GD += delta_R_sample
            batch_Sprvsr.calc_loss(batch_State)
            batch_Sprvsr.calc_grad_loss(BigClass, b)
        batch_Sprvsr.calc_loss_over_batch(0)
        batch_Sprvsr.calc_grad_loss_over_batch(0)
        delta_K_GD /= Strctr.Nin
        delta_R_GD /= Strctr.Nin
        batch_State.t = 1
        uses_update_modality = (batch_Sprvsr.training_scheme in {'BEASTAL', 'BEASTAL_NTC', 'PIECETAL_NTC'} 
                                or (batch_Sprvsr.training_scheme == 'grad_desc' and Variabs.R_update == 'deltaR_NTC'))
        if uses_update_modality:
            batch_Sprvsr.calc_update_vals_vec(BigClass)
        batch_Sprvsr.update_input(BigClass)
        batch_Sprvsr.update_output(BigClass)
        batch_State.solve_flow_given_modality(BigClass, "update", access_inters=batch_Sprvsr.access_interNodes)
        batch_State.update_Rs(BigClass)
        delta_R_BEASTAL = batch_State.R_in_t[-1] - R_before_update
        if Variabs.grad_desc_use == 'K':
            K_after_GD = np.clip(matrix_functions.K_from_R(R_before_update) + batch_Sprvsr.alpha * delta_K_GD, 1e-12, None)
            R_after_GD = 1 / K_after_GD
        elif Variabs.grad_desc_use == 'R':
            R_after_GD = np.clip(R_before_update + batch_Sprvsr.alpha * delta_R_GD, 1e-12, None)
        if Variabs.normalize_step:
            raw_delta_R_GD = R_after_GD - R_before_update
            if np.linalg.norm(raw_delta_R_GD) > 0:
                R_after_GD = R_before_update + batch_Sprvsr.alpha * raw_delta_R_GD / np.linalg.norm(raw_delta_R_GD)
        if Strctr.frozen_ground and Strctr.ground_edges.size:
            R_after_GD[Strctr.ground_edges] = R_before_update[Strctr.ground_edges]
        R_after_GD[R_after_GD < 1e-12] = 1e-12
        delta_R_GD = R_after_GD - R_before_update
    finally:
        BigClass.add_State(trained_State)
        BigClass.add_Sprvsr(trained_Sprvsr)
    return delta_R_BEASTAL, delta_R_GD

def training_loop():
    comparison_index = 0
    State.cosine_des_actual_Q = []
    State.cosine_des_actual_Q_squared = []
    for t in range(Sprvsr.iterations):
        calculate_cosine_now = Sprvsr.calculate_cosine_sim and (t == 1 or not (t+1) % Sprvsr.calculate_every)
        for b in range(Sprvsr.batch_size):
            
            # # measurement modality
            State.draw_input_and_desired(Sprvsr, Sprvsr.batch_size*t+b)
            State.solve_flow_given_modality(BigClass, "measure")

            # save R, p, u for network plots, after measurement
            if t == Sprvsr.iterations-1:
                NET.save_R_reordered(State.R_in_t[-1], Strctr.EIEJ_plots)
                NET.save_p_reordered(State.p)
                NET.save_u_reordered(State.u, Strctr.EIEJ_plots)
    #                 plot_funcs.plotNetStructure(NET.NET, BigClass, NET.pos_lattice, node_labels=False, 
    #                                     R_reordered=NET.R_reordered, u_reordered=NET.u_reordered, p_reordered=NET.p_reordered)

            Sprvsr.calc_loss(State)
            if Sprvsr.training_scheme in ['BEASTAL', 'BEASTAL_NTC', 'PIECETAL_NTC', 'grad_desc']:
                Sprvsr.calc_grad_loss(BigClass, Sprvsr.batch_size*t+b)

        Sprvsr.calc_loss_over_batch(t)
        if Sprvsr.training_scheme in ['BEASTAL', 'BEASTAL_NTC', 'PIECETAL_NTC', 'grad_desc']:
            Sprvsr.calc_grad_loss_over_batch(t)

        State.t += 1
        if not(State.t%Sprvsr.print_every):  # print time and outputs every print_every
            print('time=', State.t)
            print('measured output ', State.output)
            print('desired output ', State.desired)
            print('')

        if Sprvsr.training_scheme == 'Adjoint_pressure':
            Sprvsr.calc_adjoint_output_pressure(Strctr)
            State.solve_flow_given_modality(BigClass, modality='adjoint')
            Sprvsr.calc_update_vals_vec(BigClass)
        elif Sprvsr.training_scheme in {'Adjoint_current_noIn', 'Adjoint_pressure_noIn'}:
            Sprvsr.calc_update_vals_vec(BigClass)
        elif not((t + 1) % 4) and Sprvsr.access_interNodes:
            Sprvsr.update_inter(BigClass)
        else:  # BEASTAL etc
            if Variabs.R_update in ['deltaR_NTC', 'deltaR_propto_dp_nonlin', 'deltaR_propto_dp', 'grad_desc']:
                Sprvsr.calc_update_vals_vec(BigClass)
            Sprvsr.update_input(BigClass)  # update values
            Sprvsr.update_output(BigClass)  # update valued
            # # un-comment to not update
            # Sprvsr.input_update_nxt = Sprvsr.alpha_in_t[t]*np.ones(np.size(Strctr.input_nodes_arr))
            # Sprvsr.output_update_nxt = np.zeros(np.size(Strctr.output_nodes_arr))
            # Sprvsr.input_update_in_t.append(Sprvsr.input_update_nxt)
            # Sprvsr.output_update_in_t.append(Sprvsr.output_update_nxt)
#                     print('input_update_nxt ', Sprvsr.input_update_nxt)
#                     print('output_update_nxt ', Sprvsr.output_update_nxt)

        # delta R GD
        if calculate_cosine_now:
            Sprvsr.delta_R_BEASTAL_in_t[comparison_index], Sprvsr.delta_R_GD_in_t[comparison_index] = compare_BEASTAL_to_GD(State)

        # # update modality

        # measure and don't change resistances
        State.solve_flow_given_modality(BigClass, "update", access_inters=Sprvsr.access_interNodes)

        cosine_des_actual = np.dot(Sprvsr.desired_update_Q, State.u) / (np.linalg.norm(Sprvsr.desired_update_Q) * np.linalg.norm(State.u))
        State.cosine_des_actual_Q.append(cosine_des_actual)
        cosine_des_actual_squared = np.dot(Sprvsr.desired_update_Q**2, State.u**2) / (np.linalg.norm(Sprvsr.desired_update_Q**2) * np.linalg.norm(State.u**2))
        State.cosine_des_actual_Q_squared.append(cosine_des_actual_squared)
        
        # Resistance change due to evolution rule during update
        State.update_Rs(BigClass)  # supressed - no resistance update

        # # save resistances externally and restart them
        # R_in_t.append(State.R_in_t[-1])
        # State.R_in_t[t] = State.R_in_t[0]  # reset to initial R for next iteration
        # State.R_in_t[t+1] = State.R_in_t[0]

        # Compare actual and GD steps in the selected training parameter
        if calculate_cosine_now:
            R_before = State.R_in_t[-2]
            delta_GD = Sprvsr.delta_R_GD_in_t[comparison_index]
            if Variabs.grad_desc_use == 'K':
                delta_actual = 1/State.R_in_t[-1] - 1/R_before
                delta_GD = 1/(R_before + delta_GD) - 1/R_before
            else:  # grad_desc_use == 'R'
                delta_actual = State.R_in_t[-1] - R_before
            GD_norm, actual_norm = np.linalg.norm(delta_GD), np.linalg.norm(delta_actual)
            Sprvsr.cosine_similarity_in_t[comparison_index] = np.dot(delta_GD, delta_actual) / (GD_norm * actual_norm)
            comparison_index += 1
                
        # # anneal learning rate
        if Sprvsr.anneal_alpha_enabled and State.t < Sprvsr.iterations:
            Sprvsr.anneal_alpha(State)
        if Sprvsr.anneal_dt_enabled and State.t < Sprvsr.iterations:
            Sprvsr.anneal_dt(State, Variabs)
        if Sprvsr.anneal_stay_sample_enabled and State.t < Sprvsr.iterations:
            Sprvsr.anneal_stay_sample(State, Variabs)

#     # save R, p, u for network plots, after measurement
#     NET.save_R_reordered(State.R_in_t[-1], Strctr.EIEJ_plots)
#     NET.save_p_reordered(State.p)
#     NET.save_u_reordered(State.u, Strctr.EIEJ_plots)

    Sprvsr.calc_loss_scalar(State, Strctr)

## Set up init Rs and run

In [ ]:
base_CFG = CFG
run_settings = [  # cumulative: switch to NTCs, then remove one cheating at a time
    # ('1_grad_desc_all_cheatings', 'grad_desc', 'BEASTAL', True, True, True),  # know_R_in_evolution, know_R_in_grad_loss, access_all_edges
    # ('2_NTCs_all_cheatings', 'deltaR_NTC', 'BEASTAL_NTC', True, True, True),
    # ('3_no_R_in_grad_loss', 'deltaR_NTC', 'BEASTAL_NTC', True, False, True),
    # ('4_no_R_in_evolution', 'deltaR_NTC', 'BEASTAL_NTC', False, True, True),
    # ('5_no_R_in_evolution_and_grad', 'deltaR_NTC', 'BEASTAL_NTC', False, False, True),
    ('6_no_access_all_edges', 'deltaR_NTC', 'BEASTAL_NTC', False, False, False)]
runs = {}
for run_name, R_update, training_scheme, know_R_in_evolution, know_R_in_grad_loss, access_all_edges in run_settings:
    CFG = replace(base_CFG, Variabs=replace(base_CFG.Variabs, R_update=R_update),
                  Sprvsr=replace(base_CFG.Sprvsr, training_scheme=training_scheme, cheat_know_R_in_evolution=know_R_in_evolution, cheat_know_R_in_grad_loss=know_R_in_grad_loss,
                                 cheat_access_all_edges=access_all_edges))
    np.random.seed(base_CFG.Sprvsr.random_state)  # identical material disorder and input data in every case
    Strctr = Network_Structure(CFG)
    Strctr.build_incidence(Strctr.net_type, in_to_g=CFG.Sprvsr.control == 'current')
    Strctr.build_inverse_incidence()
    BigClass = Big_Class(Strctr)
    if Strctr.Ninter > 0:
        fict_cfg = replace(CFG, Strctr=replace(CFG.Strctr, Ninter=0, net_type='FC'))
        Strctr_fict = Network_Structure(fict_cfg)
        Strctr_fict.build_incidence(Strctr_fict.net_type)
        Strctr_fict.build_inverse_incidence()
        BigClass.add_Strctr_fict(Strctr_fict)
    Variabs = User_Variables(CFG, Strctr)
    BigClass.add_Variabs(Variabs)
    Sprvsr = Supervisor(CFG, Strctr, Variabs)
    BigClass.add_Sprvsr(Sprvsr)
    State = Network_State(CFG, BigClass)
    BigClass.add_State(State)
    NET = Networkx_Net(CFG)
    NET.buildNetwork(BigClass)
    NET.build_pos_lattice(BigClass, plot=False, node_labels=False)
    BigClass.add_NET(NET)

    R_i = State.R_from_T(BigClass, CFG.Sprvsr.initial_T)  # same initial resistances as the NTC cases
    State.initiate_resistances(BigClass, R_i.copy(), add_noise=0.0)  # NTC initialization also ignores R_noise
    if Sprvsr.control == 'current':
        if Sprvsr.cheat_know_R_in_grad_loss:  # CHEATING - you don't really know R/K for grad
            Strctr.build_current_injection(State.R_in_t[-1])
        else:  # WHAT IT ACTUALLY SHOULD BE
            Strctr.build_current_injection(Strctr.R_25)  # BEASTAL, calculated once before training
        # print('current injection matrix C', Strctr.CM)
        # print('its inverse', Strctr.CM_dagger)
    print(run_name)
    training_loop()
    if Variabs.R_update == 'deltaR_NTC':
        print('total time [s]=', np.sum(Sprvsr.dt_in_t))
    print('initial loss over 4 samples', np.mean(Sprvsr.loss_scalar_in_t[:4]))    
    print('final loss over 100 samples', np.mean(Sprvsr.loss_scalar_in_t[-100:]))
    fig_name = f'importants_{Strctr.Nin}Nin_{Strctr.Nout}Nout_alpha={Sprvsr.alpha_initial}_training_scheme={Sprvsr.training_scheme}_{run_name}.csv'
    runs[run_name] = {'CFG': CFG, 'BigClass': BigClass, 'fig_name': fig_name}
CFG = base_CFG

In [ ]:
Sprvsr.grad_loss_over_batch

In [ ]:
Sprvsr.desired_update_Q_new = Sprvsr.desired_update_Q.copy()
Sprvsr.desired_update_Q_new[2:3:]=-Sprvsr.desired_update_Q[2:3:]
print(Sprvsr.desired_update_Q)
print(Sprvsr.desired_update_Q_new)
print(CM_dagger@Sprvsr.desired_update_Q)
print(CM_dagger@Sprvsr.desired_update_Q_new)
-Sprvsr.alpha * Sprvsr.grad_loss_over_batch_norm + Sprvsr.beta

In [ ]:
print('CC_dagger', np.matmul(Strctr.CM.T, Strctr.CM_dagger.T))
print('CM_dagger', Strctr.CM_dagger)
1/Strctr.R_25
out_nodes = copy.copy(BigClass.Strctr.output_nodes_arr)
loss_for_gradient = np.asarray(BigClass.Sprvsr.loss, dtype=float).copy()
p = State.p[:Strctr.NN]
k_identical = np.ones_like(Strctr.R_25)/Strctr.R_25[0]
print('grad_loss_multi', matrix_functions.grad_loss_current(Strctr.NE, p, Strctr.DM, out_nodes, Strctr.ground_nodes_arr, loss_for_gradient, conductances=1/Strctr.R_25, parameter=BigClass.Variabs.grad_desc_use))
print('grad_loss_single', matrix_functions.grad_loss_current(Strctr.NE, p, Strctr.DM, out_nodes, Strctr.ground_nodes_arr, loss_for_gradient, conductances=k_identical, parameter=BigClass.Variabs.grad_desc_use))
# Sprvsr.Q_max
# Sprvsr.L
# print('M=', Sprvsr.M)
# print('final R teacher', Sprvsr.teacher_R/Sprvsr.teacher_R[0].ravel())
# print('final R run', runs['1_grad_desc_all_cheatings']['BigClass'].State.R_in_t[-1]/runs['1_grad_desc_all_cheatings']['BigClass'].State.R_in_t[-1][0].ravel())
# print('final R run', runs['2_NTCs_all_cheatings']['BigClass'].State.R_in_t[-1]/runs['2_NTCs_all_cheatings']['BigClass'].State.R_in_t[-1][0].ravel())
# print('final R run', runs['5_no_R_in_evolution_and_grad']['BigClass'].State.R_in_t[-1]/runs['5_no_R_in_evolution_and_grad']['BigClass'].State.R_in_t[-1][0].ravel())
# print('final R run', runs['6_no_access_all_edges']['BigClass'].State.R_in_t[-1]/runs['6_no_access_all_edges']['BigClass'].State.R_in_t[-1][0].ravel())

## Dignostics

In [ ]:
importlib.reload(plot_funcs)
importlib.reload(file_funcs)

for run_name, run in runs.items():
    BigClass, fig_name = run['BigClass'], run['fig_name']
    Strctr, Variabs, Sprvsr, State, NET = BigClass.Strctr, BigClass.Variabs, BigClass.Sprvsr, BigClass.State, BigClass.NET
    print(run_name)
    plot_funcs.plot_importants(BigClass, include_network=True, node_labels=False, save=True, name=fig_name, log_R=True, include_GD_comparison=True, include_Q_comparison=True)
    # if Sprvsr.calculate_cosine_sim and Sprvsr.cosine_t.size:
    #     comparison_name = fig_name.removesuffix('.csv').replace('importants_', 'delta_R_comparison_')
    #     comparison_video = plot_funcs.animate_delta_R_comparison(Sprvsr.cosine_t, Sprvsr.delta_R_BEASTAL_in_t, Sprvsr.delta_R_GD_in_t,
    #                                                              Sprvsr.plot_every, comparison_name)
    #     print('saved', comparison_video)
    file_funcs.export_importants_csv(BigClass, fig_name)

In [ ]:
Strctr.R_25 * np.exp(Strctr.B * (1/Sprvsr.T_max - 1/Variabs.T_room))

In [ ]:
reference_name_1 = "alternating_no_annealing" 
reference_name_2 = "alternating" 
reference_name_3 = "random" 
reference_name_4 = "random_batch_1" 

reference_base_folder = Path(r"C:\Users\roiee\OneDrive - huji.ac.il\NETWORK SIM\data\Nov11_trimmingGD_loop\6options")
reference_folder_1 = reference_base_folder / reference_name_1
reference_folder_2 = reference_base_folder / reference_name_2
reference_folder_3 = reference_base_folder / reference_name_3
reference_folder_4 = reference_base_folder / reference_name_4
reference_runs_1 = {}
reference_runs_2 = {}
reference_runs_3 = {}
reference_runs_4 = {}
comparison_rows = []

for run_name, run in runs.items():
    filename = f'importants*_{run_name}.csv'
    reference_df_1 = file_funcs.read_importants_csv(list(reference_folder_1.glob(f'importants*_{run_name}.csv'))[0])
    reference_df_2 = file_funcs.read_importants_csv(list(reference_folder_2.glob(f'importants*_{run_name}.csv'))[0])
    reference_df_3 = file_funcs.read_importants_csv(list(reference_folder_3.glob(f'importants*_{run_name}.csv'))[0])
    reference_df_4 = file_funcs.read_importants_csv(list(reference_folder_4.glob(f'importants*_{run_name}.csv'))[0])
    reference_runs_1[run_name] = reference_df_1
    reference_runs_2[run_name] = reference_df_2
    reference_runs_3[run_name] = reference_df_3
    reference_runs_4[run_name] = reference_df_4
    nout = run['BigClass'].Strctr.Nout
    reference_losses_1 = []
    reference_losses_2 = []
    reference_losses_3 = []
    reference_losses_4 = []
    for loss, mse in zip(reference_df_1['loss'].iloc[-nout:], reference_df_1['MSE'].iloc[-nout:]):
        measurement_losses = np.mean(np.square(np.asarray(loss, dtype=float)), axis=0)
        batch_mean = np.mean(measurement_losses)
        reference_losses_1.extend(measurement_losses * (mse / batch_mean if batch_mean > 0 else 0.0))
    for loss, mse in zip(reference_df_2['loss'].iloc[-nout:], reference_df_2['MSE'].iloc[-nout:]):
        measurement_losses = np.mean(np.square(np.asarray(loss, dtype=float)), axis=0)
        batch_mean = np.mean(measurement_losses)
        reference_losses_2.extend(measurement_losses * (mse / batch_mean if batch_mean > 0 else 0.0))
    for loss, mse in zip(reference_df_3['loss'].iloc[-nout:], reference_df_3['MSE'].iloc[-nout:]):
        measurement_losses = np.mean(np.square(np.asarray(loss, dtype=float)), axis=0)
        batch_mean = np.mean(measurement_losses)
        reference_losses_3.extend(measurement_losses * (mse / batch_mean if batch_mean > 0 else 0.0))
    for loss, mse in zip(reference_df_4['loss'].iloc[-nout:], reference_df_4['MSE'].iloc[-nout:]):
        measurement_losses = np.mean(np.square(np.asarray(loss, dtype=float)), axis=0)
        batch_mean = np.mean(measurement_losses)
        reference_losses_4.extend(measurement_losses * (mse / batch_mean if batch_mean > 0 else 0.0))
    reference_losses_1 = np.asarray(reference_losses_1[-nout:])
    reference_losses_2 = np.asarray(reference_losses_2[-nout:])
    reference_losses_3 = np.asarray(reference_losses_3[-nout:])
    reference_losses_4 = np.asarray(reference_losses_4[-nout:])
    reference_mse_1 = np.mean(reference_losses_1)
    reference_std_1 = np.std(reference_losses_1)
    reference_mse_2 = np.mean(reference_losses_2)
    reference_std_2 = np.std(reference_losses_2)
    reference_mse_3 = np.mean(reference_losses_3)
    reference_std_3 = np.std(reference_losses_3)
    reference_mse_4 = np.mean(reference_losses_4)
    reference_std_4 = np.std(reference_losses_4)
    current_mse = np.mean(run['BigClass'].Sprvsr.loss_scalar_in_t_over_batch[-nout:])
    current_std = np.std(run['BigClass'].Sprvsr.loss_scalar_in_t_over_batch[-nout:])
    comparison_rows.append({'run_name': run_name, f'reference_final_MSE_{reference_name_1}': reference_mse_1, f'reference_final_MSE_std_{reference_name_1}': reference_std_1, 
                            f'reference_final_MSE_{reference_name_2}': reference_mse_2, f'reference_final_MSE_std_{reference_name_2}': reference_std_2, 
                            f'reference_final_MSE_{reference_name_3}': reference_mse_3, f'reference_final_MSE_std_{reference_name_3}': reference_std_3, 
                            f'reference_final_MSE_{reference_name_4}': reference_mse_4, f'reference_final_MSE_std_{reference_name_4}': reference_std_4, 
                            'current_final_MSE': current_mse, 'current_final_STD': current_std})
final_MSE_comparison = pd.DataFrame(comparison_rows).set_index('run_name')
final_MSE_comparison.to_csv('final_MSE_comparison.csv')


In [ ]:
comparison_rows

In [ ]:
print('teacher R', Sprvsr.teacher_R)
print('actual R', State.R_in_t[-1])

In [ ]:
final_losses = [run['BigClass'].Sprvsr.loss_scalar_in_t[-run['BigClass'].Strctr.Nout:] for run in runs.values()]
initial_losses = [run['BigClass'].Sprvsr.loss_scalar_in_t[:run['BigClass'].Strctr.Nout] for run in runs.values()]
final_loss_mean = np.array([np.mean(loss) for loss in final_losses])
final_loss_std = np.array([np.std(loss) for loss in final_losses])
# initial_loss_mean = np.array([np.mean(loss) for loss in initial_losses])
initial_loss_mean = np.array([7.24 for loss in initial_losses])
initial_loss_std = np.array([np.std(loss) for loss in initial_losses])
# run_indices = np.arange(1, len(runs) + 1)
run_indices = ["GD", "NTC", "no_k_grad", "no_k_evol", "no_k_both", "boundary"]
fig, ax = plt.subplots(figsize=(5, 5))
# ax.errorbar(run_indices, final_loss_mean, yerr=final_loss_std, fmt='o', color=colors_lst[2], capsize=4, label='batch 1')
ax.plot(run_indices, initial_loss_mean, '--', color=colors_lst[3])
ax.errorbar(run_indices, final_MSE_comparison.loc[list(runs), f'reference_final_MSE_{reference_name_1}'], yerr=final_MSE_comparison.loc[list(runs), f'reference_final_MSE_std_{reference_name_1}'], fmt='o', color=colors_lst[0], capsize=4, label=reference_name_1)
# ax.errorbar(run_indices, final_MSE_comparison.loc[list(runs), f'reference_final_MSE_{reference_name_2}'], yerr=final_MSE_comparison.loc[list(runs), f'reference_final_MSE_std_{reference_name_2}'], fmt='o', color=colors_lst[1], capsize=4, label=reference_name_2)
# ax.errorbar(run_indices, final_MSE_comparison.loc[list(runs), f'reference_final_MSE_{reference_name_3}'], yerr=final_MSE_comparison.loc[list(runs), f'reference_final_MSE_std_{reference_name_3}'], fmt='o', color=colors_lst[2], capsize=4, label=reference_name_3)
# ax.errorbar(run_indices, final_MSE_comparison.loc[list(runs), f'reference_final_MSE_{reference_name_4}'], yerr=final_MSE_comparison.loc[list(runs), f'reference_final_MSE_std_{reference_name_4}'], fmt='o', color=colors_lst[3], capsize=4, label=reference_name_4)
ax.legend()
ax.set_xticks(run_indices)
ax.tick_params(axis='x', labelrotation=90)
# ax.set_xlabel('Setting index')
ax.set_ylabel('Final average loss')
ax.set_yscale('log')
ax.set_ylim([8e-5, 1.5e1])
plt.tight_layout()
plt.show()

In [ ]:
Sprvsr.loss_scalar_in_t

In [ ]:
if Sprvsr.calculate_cosine_sim:
    R_comparison = State.R_in_t[-1].copy()
    delta_R, delta_R_GD = compare_BEASTAL_to_GD(State)

### Compare the batch update with a gradient-descent step

In [ ]:
if Sprvsr.calculate_cosine_sim:
    if Variabs.grad_desc_use == 'K':
        delta, delta_GD = 1/(R_comparison + delta_R) - 1/R_comparison, 1/(R_comparison + delta_R_GD) - 1/R_comparison
    else:  # grad_desc_use == 'R'
        delta, delta_GD = delta_R, delta_R_GD
    delta_norm, delta_GD_norm = np.linalg.norm(delta), np.linalg.norm(delta_GD)
    cosine_GD = np.dot(delta, delta_GD) / (delta_norm * delta_GD_norm) if delta_norm > 0 and delta_GD_norm > 0 else np.nan
    print(f'||delta_{Variabs.grad_desc_use}|| =', delta_norm, f'||delta_{Variabs.grad_desc_use}_GD|| =', delta_GD_norm)
    print(f'cosine similarity in {Variabs.grad_desc_use} =', cosine_GD)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(delta / delta_norm if delta_norm > 0 else delta, 'o-', label='batch update')
    ax.plot(delta_GD / delta_GD_norm if delta_GD_norm > 0 else delta_GD, 'o--', label='gradient descent')
    ax.axhline(0, color='k', linewidth=1.2)
    ax.set(xlabel='edge', ylabel=rf'$\Delta {Variabs.grad_desc_use} / ||\Delta {Variabs.grad_desc_use}||_2$')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
import sys
sys.exit("program complete, Exiting kernel.")

## Iterate holding time

In [ ]:
# dt_list = np.array([1e-3, 5e-3, 1e-2, 3e-2, 6e-2, 1e-1, 1])
# R_final_per_run = []
# loss_final_per_run = []

# for i, dt in enumerate(dt_list):
#     # Variabs.euler_steps = steps
#     Variabs.dt = dt
#     Variabs.dt_upper = dt
#     State = Network_State(CFG, BigClass)
#     State.initiate_resistances(BigClass, CFG.State.R_vec_i, add_noise=CFG.State.R_noise)
#     State.initiate_accuracy_vec(BigClass)
#     BigClass.add_State(State)
#     training_loop()
#     fig_name = r"importants_{0}Nin_{1}Nout_dt={2}.csv".format(Strctr.Nin, Strctr.Nout, Variabs.dt)
#     plot_funcs.plot_importants(BigClass, include_network=True, node_labels=False, save=True, name=fig_name,
#                                 log_R=True) 
#     file_funcs.export_importants_csv(BigClass, fig_name)
#     R_final_per_run.append(State.R_in_t[-1])
#     loss_final_per_run.append(np.mean(Sprvsr.loss_scalar_in_t[-100:]))

In [ ]:
# R_final_per_run_arr = np.asarray(R_final_per_run)
# R_diff = np.diff(R_final_per_run_arr, axis=0)
# relative_err = np.sqrt(np.sum(np.square(R_diff), axis=1))/np.sqrt(np.sum(np.square(R_final_per_run_arr[-1])))
# plt.semilogy(dt_list[1:], relative_err)
# plt.ylabel('relative error')
# plt.xlabel('steps')
# plt.show()

# plt.loglog(dt_list, loss_final_per_run)
# plt.ylabel('average final loss')
# plt.xlabel(r'$dt$')
# plt.show()


In [ ]:
# import sys
# sys.exit("program complete, Exiting kernel.")

## Iterate alpha

In [ ]:
importlib.reload(file_funcs)

loss_final_per_run = []

alphas = np.array([0.001, 0.01, 0.03, 0.04, 0.05, 0.1, 1])
for i, alpha in enumerate(alphas):
    Sprvsr.assign_alpha(alpha, Variabs)
    State = Network_State(CFG, BigClass)
    State.initiate_resistances(BigClass, CFG.State.R_vec_i, add_noise=CFG.State.R_noise)
    State.initiate_accuracy_vec(BigClass)
    BigClass.add_State(State)
    training_loop()
    loss_final_per_run.append(np.mean(Sprvsr.loss_scalar_in_t[-100:]))
    if i == 0:
        print('total time=', np.sum(Sprvsr.dt_in_t))
    print('alpha=', alpha)
    fig_name = r"importants_{0}Nin_{1}Nout_alpha={2}.csv".format(Strctr.Nin, Strctr.Nout, Sprvsr.alpha)
    plot_funcs.plot_importants(BigClass, include_network=True, node_labels=False, save=True, name=fig_name, log_R=True)
    file_funcs.export_importants_csv(BigClass, fig_name)


In [ ]:
loss_final_per_run

In [ ]:
R_final_per_run_arr = np.asarray(R_final_per_run)
R_diff = np.diff(R_final_per_run_arr, axis=0)
relative_err = np.sqrt(np.sum(np.square(R_diff), axis=1))/np.sqrt(np.sum(np.square(R_final_per_run_arr[-1])))
plt.semilogy(dt_list[1:], relative_err)
plt.ylabel('relative error')
plt.xlabel('steps')
plt.show()

plt.loglog(alphas, loss_final_per_run)
plt.ylabel('average final loss')
plt.xlabel(r'$dt$')
plt.show()


## Plot importants for NTC specifically

In [ ]:
loss_in_t_array = np.reshape(np.asarray(Sprvsr.loss_in_t), (len(Sprvsr.loss_in_t), -1))
loss_derivative = np.diff(np.asarray(Sprvsr.loss_scalar_in_t))
degradation_indices = np.flatnonzero((loss_derivative[:-1] <= 0) & (loss_derivative[1:] > 0)) + 1
degradation_t = degradation_indices[0] if degradation_indices.size else None

fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(7, 9), sharex=True)
ax1.set_title(rf'$\alpha = {Sprvsr.alpha_in_t[0]:.3g}$')
ax1.plot(Sprvsr.input_update_in_t)
ax1.plot(Sprvsr.output_update_in_t)
ax1.set_ylabel('update')
ax2.plot(np.matmul(Strctr.DM, Sprvsr.update_vec_in_t.T).T[:, :])
ax2.set_ylabel(r'$\Delta p^{\,!}$')
ax3.plot(State.R_in_t)
ax3.set_ylabel(r'$R$')
ax4.plot(Sprvsr.loss_scalar_in_t)
ax4.plot(loss_in_t_array)
# ax4.set_ylim([-0.1,0.1])
ax4.axhline(0, color='k', linestyle='--', linewidth=1, alpha=0.5)
ax4.set_ylabel(r'$\mathcal{L}$')
ax5.plot(State.T_in_t)
ax5.set_ylim([290, 400])
ax5.set_ylabel(r'$T$')
if degradation_t is not None:
    for ax in (ax1, ax2, ax3, ax4, ax5):
        ax.axvline(degradation_t, color='k', linestyle=':', linewidth=1.2, alpha=0.7)
# ax4.set_ylim([998, 1010])
# ax4.set_xlabel(r'$t$')
ax5.set_xlabel(r'$t$')


plt.show()

In [ ]:
import sys
sys.exit("program complete, Exiting kernel.")

# Save sizes to file

In [ ]:
importlib.reload(file_funcs)
save_folder_prelim = ''
# save_folder_prelim = 'C:/Users/roiee/OneDrive - huji.ac.il/PhD/Network Simulation repo/Network combine/Network_combine/'

# # for general regression

# np.save(save_folder_prelim + 't.npy', State.t)
# np.save(save_folder_prelim + 'M.npy', M_values)
# np.save(save_folder_prelim + 'output_lin.npy', np.asarray(State.output_in_t)/np.asarray(Sprvsr.desired_in_t)-1)
# np.save(save_folder_prelim + 'input_update_lin.npy', Sprvsr.input_update_in_t)
# np.save(save_folder_prelim + 'output_update_lin.npy', Sprvsr.output_update_in_t)
# np.save(save_folder_prelim + 'R_lin.npy', State.R_in_t)
# np.save(save_folder_prelim + 'loss_lin.npy', Sprvsr.loss_in_t)
file_funcs.export_importants_csv(BigClass, save_folder_prelim + r"importants_nonlin_{0}_{1}_{2}.csv".format(Strctr.Nin, Strctr.Ninter, Strctr.Nout))

# dump everythin
# import pickle
# # NET.NET networkx graph
# with open(save_folder_prelim + 'NETgraph_4in6out.pkl', 'wb') as f:
#     pickle.dump(NET.NET, f)
    
# # Save the dictionary to a file
# with open(save_folder_prelim + 'pos_lattice_4in6out.pkl', 'wb') as f:
#     pickle.dump(NET.pos_lattice, f)

# # for performance_2_networks
      
# np.save(save_folder_prelim + 'output_4in6out.npy', np.asarray(State.output_in_t))
# np.save(save_folder_prelim + 'input_update_4in6out.npy', Sprvsr.input_update_in_t)
# np.save(save_folder_prelim + 'output_update_4in6out.npy', Sprvsr.output_update_in_t)
# np.save(save_folder_prelim + 'R_4in6out_Adalike.npy', State.R_in_t)
# np.save(save_folder_prelim + 'loss_scalar_4in6out_Adalike.npy', Sprvsr.loss_scalar_in_t)

# import pickle
# # NET.NET networkx graph
# with open(save_folder_prelim + 'NETgraph_1in2out.pkl', 'wb') as f:
#     pickle.dump(NET.NET, f)
    
# # Save the dictionary to a file
# with open(save_folder_prelim + 'pos_lattice_1in2out.pkl', 'wb') as f:
#     pickle.dump(NET.pos_lattice, f)

# np.save(save_folder_prelim + 'cosine_sim_4in6out.npy', cosine_vec)

# # for classification_1_material
# np.save(save_folder_prelim + 't.npy', State.t)
# if Sprvsr.task_type == 'Iris_classification':
#     np.save(save_folder_prelim + 'deltaR_ptopto_Poweraccuracy_in_t.npy', np.asarray(accuracy_in_t))
# np.save(save_folder_prelim + 'dataset_shape.npy', np.shape(Sprvsr.dataset))
# np.save(save_folder_prelim + 't_for_accuracy.npy', State.t_for_accuracy)

# for classification_5_material
# np.save(save_folder_prelim + 't.npy', State.t)
# np.save(save_folder_prelim + 'deltaR_propto_dpaccuracy_in_t.npy', accuracy_in_t)
# np.save(save_folder_prelim + 'dataset_shape.npy', np.shape(Variabs.dataset))
# np.save(save_folder_prelim + 't_for_accuracy.npy', State.t_for_accuracy)

In [ ]:
import sys
sys.exit("program complete, Exiting kernel.")

# Not In Use

## Training loop for Iris classification

In [ ]:
# State.initiate_resistances(BigClass, CFG.State.R_vec_i)
# State.initiate_accuracy_vec(BigClass)
# update_iters = 1

# def training_loop():
#     for t in range(Sprvsr.iterations):

#         # print('starting iteration', t)

#         if t % np.shape(Sprvsr.X_train)[0] == 0 and Sprvsr.task_type == 'Iris_classification':
#             State.assign_targets_Iris(BigClass)
#             # print('targets_mat', State.targets_mat)


#         for l in range(1):  # Repeat the block l times
#             # # measurement modality
#             # draw input and desired outputs from dataset and solve flow
#             State.draw_input_and_desired(Sprvsr, t)
#             State.solve_flow_given_modality(BigClass, "measure")


#             # save R, p, u for network plots, after measurement
#             if t == Sprvsr.iterations-1:
#                 NET.save_R_reordered(State.R_in_t[-1], Strctr.EIEJ_plots)
#                 NET.save_p_reordered(State.p)
#                 NET.save_u_reordered(State.u, Strctr.EIEJ_plots)
# #                 plot_funcs.plotNetStructure(NET.NET, BigClass, NET.pos_lattice, node_labels=False, 
# #                                     R_reordered=NET.R_reordered, u_reordered=NET.u_reordered, p_reordered=NET.p_reordered)

#             State.t += 1
#             if not(State.t%Sprvsr.print_every):  # print time and outputs every print_every
#                 print('time=', State.t)
#                 print('measured output ', State.output)
#                 print('desired output ', State.desired)
#                 print('')
#             Sprvsr.calc_loss(State)

#             if Sprvsr.training_scheme == 'Adjoint_pressure':
#                 Sprvsr.calc_adjoint_output_pressure(Strctr)
#                 State.solve_flow_given_modality(BigClass, modality='adjoint')
#                 Sprvsr.calc_update_vals_vec(BigClass)
#             elif Sprvsr.training_scheme in {'Adjoint_current_noIn', 'Adjoint_pressure_noIn'}:
#                 Sprvsr.calc_update_vals_vec(BigClass)
#             elif not((t + 1) % 4) and Sprvsr.access_interNodes:
#                 Sprvsr.update_inter(BigClass)
#             else:  # BEASTAL etc
#                 if Sprvsr.training_scheme in ['BEASTAL', 'BEASTAL_NTC', 'PIECETAL_NTC']:
#                     Sprvsr.calc_update_vals_vec(BigClass)
#                 Sprvsr.update_input(BigClass)  # update values
#                 Sprvsr.update_output(BigClass)  # update valued
#                 # # un-comment to not update
#                 # Sprvsr.input_update_nxt = Sprvsr.alpha_in_t[t]*np.ones(np.size(Strctr.input_nodes_arr))
#                 # Sprvsr.output_update_nxt = np.zeros(np.size(Strctr.output_nodes_arr))
#                 # Sprvsr.input_update_in_t.append(Sprvsr.input_update_nxt)
#                 # Sprvsr.output_update_in_t.append(Sprvsr.output_update_nxt)
# #                     print('input_update_nxt ', Sprvsr.input_update_nxt)
# #                     print('output_update_nxt ', Sprvsr.output_update_nxt)

#                 # # update modality
#                 for m in range(update_iters):
#                     # measure and don't change resistances
#                     State.solve_flow_given_modality(BigClass, "update", access_inters=Sprvsr.access_interNodes)

#                     # print("actual update Q =", State.u)

#                     State.update_Rs(BigClass)  # supressed - no resistance update

#                     # # save resistances externally and restart them
#                     # R_in_t.append(State.R_in_t[-1])
#                     # State.R_in_t[t] = State.R_in_t[0]  # reset to initial R for next iteration
#                     # State.R_in_t[t+1] = State.R_in_t[0]
                        
#                 # # anneal learning rate
#                 if Sprvsr.anneal_alpha_enabled and State.t < Sprvsr.iterations:
#                     Sprvsr.anneal_alpha(State)
#                 if Sprvsr.anneal_dt_enabled and State.t < Sprvsr.iterations:
#                     Sprvsr.anneal_dt(State, Variabs)
#                 if Sprvsr.anneal_stay_sample_enabled and State.t < Sprvsr.iterations:
#                     Sprvsr.anneal_stay_sample(State, Variabs)
#         # print('due to update_vec=', Sprvsr.update_vec)
#         # print('resulting in resistance=', State.R_in_t[-1])
#         # measure accuracy in classification task
#         if t % Sprvsr.measure_accuracy_every == 0 and Sprvsr.task_type == 'Iris_classification' \
#             and t//Sprvsr.measure_accuracy_every<len(State.accuracy_in_t):
#             if t==0:
#                 State.accuracy_in_t[t] = 1/3
#                 State.t_for_accuracy[t//Sprvsr.measure_accuracy_every] = State.t 
#             else:
#                 State.calculate_accuracy_testset(BigClass)
#                 State.accuracy_in_t[t//Sprvsr.measure_accuracy_every] = State.accuracy 
#                 State.t_for_accuracy[t//Sprvsr.measure_accuracy_every] = State.t 
# #     # save R, p, u for network plots, after measurement
# #     NET.save_R_reordered(State.R_in_t[-1], Strctr.EIEJ_plots)
# #     NET.save_p_reordered(State.p)
# #     NET.save_u_reordered(State.u, Strctr.EIEJ_plots)

#     Sprvsr.calc_loss_scalar(State, Strctr)

In [ ]:
# if Sprvsr.task_type == 'Iris_classification':
#     accuracy_in_t = []
#     random_state = CFG.Sprvsr.random_state
#     for i in range(8):
#         random_state += 1
#         if random_state == 56:
#             random_state += 1
#         elif random_state == 61:
#             random_state += 1
#         print('random_state = ', random_state)
#         iris_cfg = replace(CFG, Sprvsr=replace(CFG.Sprvsr, random_state=random_state))
#         Sprvsr.create_dataset_and_targets(iris_cfg, Strctr, Variabs, train_size=0.2)
#         Sprvsr.create_noise_for_extras(Strctr, Variabs)
#         State = Network_State(CFG, BigClass)
#         State.initiate_resistances(BigClass, CFG.State.R_vec_i)
#         State.initiate_accuracy_vec(BigClass)
#         BigClass.add_State(State)  # add to big class
#         training_loop()
#         # plot_funcs.plot_importants(BigClass, movmean_loss = True, include_network=False, node_labels=False)
#         accuracy_in_t.append(State.accuracy_in_t)
#         # plot_funcs.plot_accuracy(State.t, State.t_for_accuracy, State.accuracy_in_t, np.shape(Variabs.dataset)[0])

In [ ]:
# if Sprvsr.task_type == 'Iris_classification':
#     plot_funcs.plot_accuracy(State.t, State.t_for_accuracy, State.accuracy_in_t, np.shape(Sprvsr.dataset)[0])
# else:
#     pass

## Training loop for GD style

In [ ]:
# GD_cost_vec = []
# dK_arr = []
# dK_GD_arr = []
# cosine_vec: list[float] = []
# accuracy_in_t: list[np.ndarray] = []
# cosine_lin = []
# cosine_nonlin = []
# cosine_optimal = []
# R_in_t = []
# dK_step=10**(-8)

# def training_loop():
#     for t in range(Sprvsr.iterations):

#         # print('starting iteration', t)

#         for l in range(1):  # Repeat the block l times
#             # # measurement modality
#             # draw input and desired outputs from dataset and solve flow
#             State.draw_input_and_desired(Sprvsr, t)
#             State.solve_flow_given_modality(BigClass, "measure")


#             if Sprvsr.calculate_cosine_sim or Variabs.R_update == 'grad_desc':
#                 # GD cost
#                 State.calc_GD_cost(Strctr, State.desired, mod='measure', func='MSE')
#                 GD_cost_vec.append(State.GD_cost)

#                 # change in K grad desc
#                 delta_K_GD, dcost_dK = State.dK_grad_desc(Strctr, dK_step, State.desired, func='MSE')   
#                 dK_GD_arr.append(delta_K_GD)

#             # save R, p, u for network plots, after measurement
#             if t == Sprvsr.iterations-1:
#                 NET.save_R_reordered(State.R_in_t[-1], Strctr.EIEJ_plots)
#                 NET.save_p_reordered(State.p)
#                 NET.save_u_reordered(State.u, Strctr.EIEJ_plots)
# #                 plot_funcs.plotNetStructure(NET.NET, BigClass, NET.pos_lattice, node_labels=False, 
# #                                     R_reordered=NET.R_reordered, u_reordered=NET.u_reordered, p_reordered=NET.p_reordered)

#             if Sprvsr.include_Power:
#                 State.calc_Power_norm(BigClass)

#             if not t % 2 and Sprvsr.use_p_tag:  # even iterations, take another sampled pressure and measure again
#                 pass
#             else:  # odd iterations, go to update modality and update resistances
#                 if Strctr.net_type == 'beads':  # if beads - need a few iterations to converge
#                     update_iters = 6
#                 else:  # if not beads, converges within a single iteration
#                     update_iters = 1

#                 State.t += 1
#                 if not(State.t%Sprvsr.print_every):  # print time and outputs every print_every
#                     print('time=', State.t)
#                     print('measured output ', State.output)
#                     print('desired output ', State.desired)
#                     print('')
#                 Sprvsr.calc_loss(State)

#                 if Sprvsr.training_scheme == 'Adjoint_pressure':
#                     Sprvsr.calc_adjoint_output_pressure(Strctr)
#                     State.solve_flow_given_modality(BigClass, modality='adjoint')
#                     Sprvsr.calc_update_vals_vec(BigClass)
#                 elif Sprvsr.training_scheme in {'Adjoint_current_noIn', 'Adjoint_pressure_noIn'}:
#                     Sprvsr.calc_update_vals_vec(BigClass)
#                 elif not((t + 1) % 4) and Sprvsr.access_interNodes:
#                     Sprvsr.update_inter(BigClass)
#                 else:  # BEASTAL etc
#                     if Sprvsr.training_scheme in ['BEASTAL', 'BEASTAL_NTC', 'PIECETAL_NTC']:
#                         Sprvsr.calc_update_vals_vec(BigClass)
#                     Sprvsr.update_input(BigClass)  # update values
#                     Sprvsr.update_output(BigClass)  # update valued
#                     # # un-comment to not update
#                     # Sprvsr.input_update_nxt = Sprvsr.alpha_in_t[t]*np.ones(np.size(Strctr.input_nodes_arr))
#                     # Sprvsr.output_update_nxt = np.zeros(np.size(Strctr.output_nodes_arr))
#                     # Sprvsr.input_update_in_t.append(Sprvsr.input_update_nxt)
#                     # Sprvsr.output_update_in_t.append(Sprvsr.output_update_nxt)
# #                     print('input_update_nxt ', Sprvsr.input_update_nxt)
# #                     print('output_update_nxt ', Sprvsr.output_update_nxt)

#                 # # update modality
#                 for m in range(update_iters):
#                     # measure and don't change resistances
#                     State.solve_flow_given_modality(BigClass, "update", access_inters=Sprvsr.access_interNodes)

#                     # print("actual update Q =", State.u)

#                     State.update_Rs(BigClass)  # supressed - no resistance update

#                     # # save resistances externally and restart them
#                     # R_in_t.append(State.R_in_t[-1])
#                     # State.R_in_t[t] = State.R_in_t[0]  # reset to initial R for next iteration
#                     # State.R_in_t[t+1] = State.R_in_t[0]
                        
#                 # # anneal learning rate
#                 if Sprvsr.anneal_alpha_enabled and State.t < Sprvsr.iterations:
#                     Sprvsr.anneal_alpha(State)
#                 if Sprvsr.anneal_dt_enabled and State.t < Sprvsr.iterations:
#                     Sprvsr.anneal_dt(State, Variabs)
#                 if Sprvsr.anneal_stay_sample_enabled and State.t < Sprvsr.iterations:
#                     Sprvsr.anneal_stay_sample(State, Variabs)
#         # print('due to update_vec=', Sprvsr.update_vec)
#         # print('resulting in resistance=', State.R_in_t[-1])

# #     # save R, p, u for network plots, after measurement
# #     NET.save_R_reordered(State.R_in_t[-1], Strctr.EIEJ_plots)
# #     NET.save_p_reordered(State.p)
# #     NET.save_u_reordered(State.u, Strctr.EIEJ_plots)

#     Sprvsr.calc_loss_scalar(State, Strctr)

## Power consumption

In [ ]:
# if np.size(State.Power_norm_in_t)>1:
#     plot_funcs.plot_Power(State)
# else:
#     pass

## Power of trained network

put a pressure of 1 through all inputs and measure total power dissipation in a trained network that has the state State

In [ ]:
# # Reload the module to reflect any changes made
# importlib.reload(statistics)

# # put pressure of 1 through inputs
# State.input_drawn = np.ones(Nin)

# # solve flow
# State.solve_flow_given_modality(BigClass, "measure")

# # measure power
# print('u', State.u)
# print('Rs', State.R_in_t[-1])
# print('Power dissipation', statistics.power_dissip(State.u, State.R_in_t[-1]))

## comparison Rs GD and Adalike

In [ ]:
# import matplotlib.gridspec as gridspec

# # Set color cycle globally
# plt.rcParams['axes.prop_cycle'] = plt.cycler('color', Colorscheme.colors_lst)

# # Normalize R values so maximal will be 1
# R_GD_norm = R_GD / np.max(R_GD)
# R_ourscheme_norm = R_ourscheme / np.max(R_ourscheme)

# # R_GD_norm = np.concatenate([R_GD_norm[:-1], R_GD_norm[-1:]])
# # R_ourscheme_norm = np.concatenate([R_ourscheme_norm[:-1],  R_ourscheme_norm[-1:]])

# # weird setup for bars
# x = np.arange(len(R_ourscheme_norm))
# bar_width = 0.35

# # instantiate figure and grid for positioning colorbal
# fig = plt.figure(figsize=(7, 3))
# gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1], wspace=0.45)

# # R bar plot
# ax0 = fig.add_subplot(gs[0, 0])
# ax0.bar(x - bar_width / 2, R_GD_norm,
#         width=bar_width, label='GD', alpha=0.8, edgecolor='k', linewidth=1.6)
# ax0.bar(x + bar_width / 2, R_ourscheme_norm,
#         width=bar_width, label='this work', alpha=0.8, edgecolor='k', linewidth=1.6)
# ax0.set_xticks([0, 1, 2, 3])
# ax0.set_yticks([0, 0.5, 1])
# ax0.set_ylabel('$R$')
# # ax0.legend()

## Is the network linear?

In [ ]:
# # put pressure of 1 through 1st input
# State.input_drawn = np.array([1,0])

# # solve flow
# State.solve_flow_given_modality(BigClass, "measure")

# # measure power
# print('output 1', State.output)
# out1 = State.output

# # put pressure of 1 through 1st input
# State.input_drawn = np.array([0,1])

# # solve flow
# State.solve_flow_given_modality(BigClass, "measure")

# # measure power
# print('output 2', State.output)
# out2 = State.output

# print('superpose outputs', out1+out2)

# # put pressure of 1 through inputs
# State.input_drawn = np.ones(Nin)

# # solve flow
# State.solve_flow_given_modality(BigClass, "measure")

# # measure power
# print('output both', State.output)

## cosine similarity

In [ ]:
# if Sprvsr.calculate_cosine_sim:
#     plt.rcParams['axes.prop_cycle'] = plt.cycler('color', colors_lst)

#     loss_movmean = statistics.mov_ave(Sprvsr.loss_scalar_in_t, 40)

#     # Create subplots: 2 rows, 1 column
#     fig, (ax1, ax2, ax3) = plt.subplots(3, 1, sharex=True, figsize=(6, 5))

#     # First subplot: resistances
#     ax1.plot(BigClass.State.R_in_t[1:])
#     ax1.set_ylabel(r'$R$')

#     # Second subplot: Cosine similarity
#     ax2.set_prop_cycle(plt.cycler('color', colors_lst))  # set color cycle
#     ax2.plot(cosine_vec)
#     ax2.plot(statistics.mov_ave(cosine_vec, window_size=24))
#     ax2.plot(np.zeros([len(cosine_vec)]), '--k')
#     ax2.set_ylabel('Cos Similarity')
#     ax2.set_ylim([-1, 1])

#     # Third subplot: Loss moving average
#     ax3.plot(loss_movmean)
#     ax3.set_xlabel('$t$')
#     ax3.set_ylabel(r'$\|\mathcal{L}\|$')
#     ax3.set_yscale('log')
#     ax3.set_ylim(1e-4, 1e0)

#     plt.tight_layout()
#     plt.show()
    
#     plt.plot(cosine_vec)
#     plt.plot(np.zeros([len(cosine_vec)]), '--k')
#     plt.ylabel('Cos Similarity')
#     plt.ylim([-1, 1])
#     plt.xlabel('$t$')